In [ ]:
# - 모든 픽셀을 다 숫자로 변환해서 단순 숫자 계산 하는게 맞나?? => 앞에서 한 사진을 픽셀 별로 나눠서 이 칸에는 이거 저 칸에는 저거 틀에 딱 박힌 채로 계산을 하는 것이 맞나 -> 사진의 구도가 조금만 달라져도 다른 결과가 나오게 됨
# - 위치별 패텬 파악하는게 맞지 않을까? -> 머리 밑에 목, 목 믿에 몸통... => 모든 픽셀을 다 볼 필요가 없어짐

# CNN(Convolutional NN): 합성곱 신경망
#   kernel/filter로 긁어가면서
#       주변만 연결하여 결과 내서 -> 위치정보에 대한 결과
#       다양한 특성을 가진 kernel로 긁어서 channel 축으로 쌓아서 -> 이미지의 특성 
#   - padding: 데이터 바깥쪽에 추가해서 kernel로 긁어도 데이터의 사이즈가 줄지 않게 해주는 것
#   - stride: kernel이 얼마씩 움직일지
#   - (max/avg) pooling: 대푯값 뽑기 (kernel 범위 내에서 최대/평균)
# => 여러 층의 CNN을 통과하고 (위치별 특성 추출), 마지막에 FC(Fully Connected) Layer(위치별 특성 추출해놓은 것 전체)를 통과
# => 어떤 이미지가 있을 때 kernelSize? padding? strid? pooling? 각 값들을 얼마 줘야 이미지 인식을 잘 할까? : 노가다 -> 계속 바꿔가면서 적절한 값을 찾아내야함
# -> AI 연구하는 사람들이 노가다를 해서 최적의 모델들을 만들어놨음(Resnet, ...)

In [1]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset,DataLoader # PyTorch에서 제공하는 데이터 관련 라이브러리

# 이미지 파일 불러오기
import cv2
import numpy as np
import os

# 데이터를 훈련/검증용으로 나누기 위해
from random import random

In [2]:
device = "cpu"

In [7]:
# 70% 훈련용, 30% 테스트용
def getImg(folder, w, h, yData):  
    trainData, trainLabel, testData, testLabel = [], [], [],[]                    
    for i, f in enumerate(os.listdir(folder)):                # 사진 가로, 세로 = 행열
        data = cv2.imread(folder + "/" + f, cv2.IMREAD_COLOR)
        data = cv2.resize(data, (w, h))
        print(data)
        if random() < 0.3:
            testData.append(np.array(data, dtype=np.float32))
            testLabel.append(yData[i])
        else:
            trainData.append(np.array(data, dtype=np.float32))
            trainLabel.append(yData[i])
    return trainData, trainLabel, testData, testLabel

In [9]:
label = ["떡볶이", "오뎅", "김밥", "튀김", "순대"]
yData = [0,0,0,0,0, 1,1,1,1,1, 2,2,2,2,2, 3,3,3,3,3, 4,4,4,4,4]         


trainData, trainLabel, testData, testLabel = getImg("./Users/stu_ict01_01/Jun04_1_DeepLearning/bunsikMenu", 100, 50, yData)
# print(trainData[0])
# print(trainLabel[0])
# print(testData[0])
# print(testLabel[0])


[[221 221 220 ... 219 219 219]
 [220 221 219 ... 218 219 218]
 [220 221 219 ... 218 219 218]
 ...
 [218 218 218 ... 219 221 221]
 [218 220 220 ... 220 220 223]
 [220 219 219 ... 220 221 223]]
[[218 218 218 ... 217 218 218]
 [218 218 218 ... 218 218 218]
 [218 218 218 ... 218 218 218]
 ...
 [219 219 219 ... 216 216 216]
 [219 219 219 ... 216 216 216]
 [219 219 219 ... 217 216 216]]
[[216 215 216 ... 217 215 215]
 [216 216 215 ... 218 217 217]
 [216 216 215 ... 218 217 217]
 ...
 [216 216 218 ... 215 213 213]
 [215 217 217 ... 215 214 214]
 [216 217 216 ... 215 213 213]]
[[214 215 215 ... 213 213 213]
 [214 214 214 ... 213 213 213]
 [214 214 214 ... 213 213 213]
 ...
 [215 217 217 ... 213 213 213]
 [215 215 215 ... 213 213 213]
 [214 215 215 ... 213 213 213]]
[[212 212 212 ... 209 209 209]
 [212 213 213 ... 209 209 209]
 [212 213 213 ... 209 209 209]
 ...
 [209 209 209 ... 207 206 207]
 [209 209 209 ... 207 206 207]
 [209 209 209 ... 207 207 209]]
[[216 216 215 ... 220 220 220]
 [217 217

In [ ]:
# PyTorch의 Dataset + DataLoader
class BunsikDS(Dataset):
    def __init__(self, data, label, numClasses):
        super().__init__()
        self.datas = torch.from_numpy(np.array(data))       # 17개 * 세로50 * 가로100 * 색3
        self.datas = self.datas.permute(0, 3, 1, 2)         # 17*50*100*3 -> 17*3*50*100
        self.labels = torch.from_numpy(np.array(label))
        self.labels = nn.functional.one_hot(self.labels, numClasses)
        self.labels = self.labels.type(torch.float32)
        

    def __len__(self):                              # 데이터 전체 갯수 리턴
        return len(self.datas)

    def __getitem__(self, index):                   # (데이터, 라벨) 튜플 형태로 리턴
        return (self.datas[index], self.labels[index])

trainDS = BunsikDS(trainData, trainLabel, 5)        # 떡볶이, 김밥, 오뎅, 순대, 튀김 5종류
testDS = BunsikDS(testData, testLabel, 5)
trainDL = DataLoader(trainDS, 3, shuffle=True)      # trainDS에서 3개씩(batch size) 순서 섞어서
testDL = DataLoader(testDS, 3, shuffle=True)

In [ ]:
class BunsikDNN(nn.Module):
    def __init__(self):
        super().__init__()                     
        self.kbcnn = nn.Sequential(         # 여러 층의 CNN
            nn.Conv2d(3, 1000, 5),          # in(R,G,B), out, 커널사이즈
            nn.ReLU(),
            nn.Conv2d(1000, 500, 5),
            nn.ReLU(),
            nn.Conv2d(500, 100, 5),
            nn.ReLU(),
            nn.Conv2d(100, 50, 5),
            nn.ReLU()
            )        
        self.f = nn.Flatten()               # CNN 통과한거 펼쳐서
        self.kbdnn = nn.Sequential(         # 마지막에 FCLayer 통과
            nn.Linear(450, 10),             # CNN 계산식에 맞게해야함 => 계산 힘드니 실행 시 에러 출력 값에 나오는 값 적으면 됨
            nn.ReLU(),
            nn.Linear(10, 5)
        )        

    def forward(self, x):    
        result = self.nbcnn(x)
        result = self.f(result)
        result = self.kbdnn(result)                  
        return result

model = MovieGenreDNN().to(device)

lossFn = torch.nn.CrossEntropyLoss()          
o = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
model.train()

for epoch in range(1000):
    for x, y in trainDL:
        x = x.to(device)
        y = y.to(device)

        predY = model(x)
        l = lossFn(predY, y)                  

        o.zero_grad()
        l.backward()
        o.step()

    print(l.item())

In [ ]:
# 정확도 테스트
# 전체 25개 중에 70% 정도 훈련 데이터, 30% 정도 테스트 데이터
# 그 테스트 테이터가 순대인데 AI가 순대라고 예측하는지 테스트
model.eval()

with torch.no_grad():                   # GD하지않음
    for x, y in testDL:
        x = x.to(device)
        y = y.to(device)
        predY = model(x)

        ok += (predY.argmax(1) == y.argmax(1)).type(torch.float32).sum().item()
    print(ok / len(testDL.dataset))     # 정답 맞춘 횟수

In [ ]:
f = cv.imread("/home/azureuser/cloudfiles/code/Users/stu_ict01_01/Jun02_1_DeepLearning/test.png", cv2.IMREAD_COLOR)
y = cv.resize(f, (100, 50))

predictData = np.array([[f]], dtype=np.float32)
predictData = torch.from_numpy(predictData)
predictData = predictData.permute(0,3,1,2)

result = model(predictData)
result = torch.nn.Softmax()(result)

print(label[result.argmax().item()])